# Xhakinha Clips AI — sessão Kaggle
Ativar: Settings → Internet **On**; Accelerator **GPU T4/P100** (opcional); Add-ons → Secrets (lista em KAGGLE_SETUP.txt).

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/SEU_USUARIO/xhakinha-clips-ai.git"   # <- troque pelo seu repositório (privado exige token nos Secrets)
if not os.path.isdir("/kaggle/working/app"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/app"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/app", "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/kaggle/working/app/requirements-kaggle.txt"], check=True)
sys.path.insert(0, "/kaggle/working/app")

In [ ]:
# VALIDAÇÃO (rápida, NÃO inicia o cérebro): confere Secrets, Turso (leitura+escrita) e esquema. Rode ANTES da célula do runner.
import subprocess, sys, os
from xhakinha.cloud.kaggle_runner import load_kaggle_secrets
print("Secrets carregados (só nomes):", load_kaggle_secrets())
r = subprocess.run([sys.executable, "/kaggle/working/app/scripts/turso_check.py"], capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
r = subprocess.run([sys.executable, "/kaggle/working/app/scripts/turso_migrate.py", "check"], capture_output=True, text=True, cwd="/kaggle/working/app")
print(r.stdout, r.stderr[-500:])

In [ ]:
from xhakinha.cloud.healthcheck import local_report
import json; print(json.dumps(local_report(), indent=2))

In [ ]:
# Roda até ~10,5 h (cérebro + executor). Só rode depois que a célula de VALIDAÇÃO acima disser "TUDO OK" e o esquema do Turso existir.
# Pode interromper a qualquer momento: o estado fica no Turso e os arquivos no Drive.
from xhakinha.cloud import kaggle_runner
kaggle_runner.main()